In [ ]:
# ============================================================
#   Hybrid QNN - Virus MNIST  |  Improved Version
#   Changes: residual block, RY+RZ quantum circuit, learnable
#   q_scale, label smoothing, cosine LR, bug fixes, stronger aug
# ============================================================

import numpy as np
import random
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.utils.class_weight import compute_class_weight
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ─────────────────────────────────────────────
#  1. REPRODUCIBILITY
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ─────────────────────────────────────────────
#  2. CONFIGURATION
# ─────────────────────────────────────────────
n_qubits    = 8
batch_size  = 16
num_classes = 10
num_epochs  = 80
lr          = 0.0003
Q_LAYERS    = 8      # increased from 6

TRAIN_PATH = 'train'
VAL_PATH   = './val'
TEST_PATH  = 'test'

# ─────────────────────────────────────────────
#  3. TRANSFORMS
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.RandomRotation(15),                              # was 10
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3), # i believe it should not be used
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    transforms.RandomErasing(p=0.3)
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ─────────────────────────────────────────────
#  4. DATASETS
# ─────────────────────────────────────────────
try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
    print("Datasets loaded successfully.")
    print(f"  Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
    
except Exception as e:
    print(f"Error loading datasets: {e}")
    raise

# ─────────────────────────────────────────────
#  5. CLASS WEIGHTS  (bug fix: class_weights -> class_weight_tensor)
# ─────────────────────────────────────────────
try:
    labels_list   = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(labels_list),
        y=labels_list
    )
    class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print("Class weights computed:", np.round(class_weights, 3))
except Exception as e:
    print(f"Could not compute class weights ({e}). Using uniform weights.")
    class_weight_tensor = torch.ones(num_classes).to(device)

# ─────────────────────────────────────────────
#  6. WEIGHTED SAMPLER + DATALOADERS
# ─────────────────────────────────────────────
labels_list        = [label for _, label in train_dataset.samples]
class_sample_counts = np.bincount(labels_list)
sample_weights     = [1.0 / class_sample_counts[label] for label in labels_list]

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)

# ─────────────────────────────────────────────
#  7. QUANTUM CIRCUIT  (RY + RZ, 8 layers)
# ─────────────────────────────────────────────
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights_ry, weights_rz):
    # Encode classical features into qubit rotations
    for i in range(n_qubits):
        qml.RX(inputs[..., i], wires=i)
        qml.RY(inputs[..., i], wires=i)

    # Variational layers with both RY and RZ (more expressive than RY alone)
    for l in range(weights_ry.shape[0]):
        for i in range(n_qubits):
            qml.RY(weights_ry[l][i], wires=i)
            qml.RZ(weights_rz[l][i], wires=i)
        # Ring entanglement
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {
    "weights_ry": (Q_LAYERS, n_qubits),
    "weights_rz": (Q_LAYERS, n_qubits)
}

# ─────────────────────────────────────────────
#  8. CLASSICAL CNN COMPONENTS
# ─────────────────────────────────────────────

class ResBlock(nn.Module):
    """
    Residual block that helps the CNN learn finer discriminative features,
    especially for confused classes like 0, 7, 8 seen in the confusion matrix.
    """
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))  # skip connection


class FeatureReduce(nn.Module):
    """
    CNN feature extractor that reduces (1,32,32) image to n_qubits values
    suitable for quantum angle encoding.
    """
    def __init__(self, final_dim, dropout=0.2):
        super().__init__()

        # Main conv stack (without final pool — pool is applied after residual)
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )

        self.res   = ResBlock(128)              # NEW: residual block
        self.pool  = nn.AdaptiveAvgPool2d((1, 1))

        # Expand-project strategy for richer quantum input encoding
        self.fc_expand  = nn.Linear(128, 64)
        self.fc_project = nn.Linear(64, final_dim)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.res(x)          # residual refinement before pooling
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.activation(self.fc_expand(x))
        x = self.fc_project(x)
        return x


# ─────────────────────────────────────────────
#  9. FULL HYBRID MODEL
# ─────────────────────────────────────────────
class HybridQNN(nn.Module):
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer           = TorchLayer(quantum_circuit, weight_shapes)
        self.q_scale           = nn.Parameter(torch.ones(n_qubits))  # NEW: learnable per-qubit scale
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x     = self.feature_extractor(x)
        x     = torch.tanh(x)          # clip to (-1,1) for angle encoding
        q_out = self.q_layer(x)
        q_out = q_out * self.q_scale   # learned amplification of qubit outputs
        return self.classifier(q_out)


# ─────────────────────────────────────────────
#  10. TRAINING & EVAL FUNCTIONS
# ─────────────────────────────────────────────
def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct = 0.0, 0

    for inputs, labels in tqdm(dataloader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = loss_fn(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (outputs.argmax(dim=1) == labels).sum().item()

    return total_loss / len(dataloader), correct / len(dataloader.dataset)


def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs        = model(inputs)
            loss           = loss_fn(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()

    return total_loss / len(dataloader), correct / total


# ─────────────────────────────────────────────
#  11. MODEL, OPTIMIZER, LOSS, SCHEDULER
# ─────────────────────────────────────────────
model     = HybridQNN(n_qubits=n_qubits, num_classes=num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.005)

# Label smoothing helps with over-confident misclassifications (class 0 issue)
loss_fn   = nn.CrossEntropyLoss(weight=class_weight_tensor, label_smoothing=0.1)

# Cosine annealing gives smoother LR decay than ReduceLROnPlateau for QML
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=num_epochs, eta_min=1e-6
)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

# ─────────────────────────────────────────────
#  12. TRAINING LOOP
# ─────────────────────────────────────────────
best_val_loss            = float('inf')
best_val_acc             = 0.0
early_stopping_patience  = 15
epochs_without_improvement = 0

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

print(f"\nStarting training for {num_epochs} epochs...")
print("=" * 60)

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss,   val_acc   = evaluate(model, val_loader, loss_fn, device)

    scheduler.step()   # cosine step (no argument needed)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1:03d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | "
          f"LR: {scheduler.get_last_lr()[0]:.2e}")

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc  = val_acc
        epochs_without_improvement = 0
        torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc
    }, "exp2_3.pth")
        print("   💾 Best model saved.")
    else:
        epochs_without_improvement += 1
        print(f"   🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"\n⏹️  Early stopping after {epoch+1} epochs.")
        break

print(f"\n✅ Best Val Loss: {best_val_loss:.4f} | Best Val Acc: {best_val_acc:.4f}")

# ─────────────────────────────────────────────
#  13. FINAL EVALUATION ON TEST SET
# ─────────────────────────────────────────────
model.load_state_dict(torch.load("exp2_3.pth"))
test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
print(f"\n🧪 Test Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.4f}")


Using device: cuda
Datasets loaded successfully.
  Train: 43585 | Val: 4837 | Test: 3458
Class weights computed: [2.069 0.674 1.71  2.173 6.554 0.78  0.337 0.691 2.019 1.555]

Model parameters: 548,538

Starting training for 80 epochs...


Epoch 001/80 | Train Loss: 1.4745  Acc: 0.2918 | Val Loss: 2.6634  Acc: 0.2109 | LR: 3.00e-04
   💾 Best model saved.


Epoch 002/80 | Train Loss: 1.0921  Acc: 0.5375 | Val Loss: 2.3325  Acc: 0.3914 | LR: 3.00e-04
   💾 Best model saved.


Epoch 003/80 | Train Loss: 0.9682  Acc: 0.6050 | Val Loss: 2.0713  Acc: 0.4112 | LR: 2.99e-04
   💾 Best model saved.


Epoch 004/80 | Train Loss: 0.9147  Acc: 0.6482 | Val Loss: 1.9520  Acc: 0.4813 | LR: 2.98e-04
   💾 Best model saved.


...

Epoch 023/80 | Train Loss: 0.6954  Acc: 0.8345 | Val Loss: 1.5417  Acc: 0.8429 | LR: 2.43e-04
   🕒 No improvement for 2 epoch(s).


Epoch 024/80 | Train Loss: 0.6912  Acc: 0.8394 | Val Loss: 1.5618  Acc: 0.8383 | LR: 2.38e-04
   🕒 No improvement for 3 epoch(s).


Epoch 025/80 | Train Loss: 0.6827  Acc: 0.8407 | Val Loss: 1.5829  Acc: 0.8110 | LR: 2.34e-04
   🕒 No improvement for 4 epoch(s).


Epoch 026/80 | Train Loss: 0.6877  Acc: 0.8380 | Val Loss: 1.5467  Acc: 0.8416 | LR: 2.29e-04
   🕒 No improvement for 5 epoch(s).


Training:  50%|████▉     | 1361/2725 [04:47<04:16,  5.31it/s]

In [ ]:
# -- Loss & Accuracy curves --

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses,   label='Val Loss')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(train_accs, label='Train Acc')
axes[1].plot(val_accs,   label='Val Acc')
axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.close()

# -- Confusion Matrix --
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.close()

# -- Classification Report --
print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

# -- t-SNE of feature extractor output --
from sklearn.manifold import TSNE

model.eval()
features_list, label_list = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        feats  = model.feature_extractor(inputs)
        features_list.append(feats.cpu().numpy())
        label_list.extend(labels.numpy())

features_np = np.concatenate(features_list, axis=0)
labels_np   = np.array(label_list)

tsne      = TSNE(n_components=2, random_state=42, perplexity=30)
reduced   = tsne.fit_transform(features_np)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(reduced[:, 0], reduced[:, 1],
                      c=labels_np, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, label='Class')
plt.title('t-SNE of Feature Extractor Output (Improved Model)')
plt.tight_layout()
plt.savefig('tsne_features.png', dpi=150)
plt.close()

print("\nAll plots saved: training_curves.png | confusion_matrix.png | tsne_features.png")